# 03 — Huấn luyện và chạy pipeline hoàn chỉnh

Notebook gốc bước 0–8 được **chuyển thành các hàm/module trong `src/`**. Notebook này gọi `run_pipeline.main()` thay vì chép toàn bộ hàng trăm dòng lần nữa. Bằng cách đó notebook và lệnh `python run_pipeline.py` chạy đúng **cùng một mã nguồn**: đọc dataset → tiền xử lý → Isolation Forest + Autoencoder → hiệu chuẩn ngưỡng → test và baseline (nếu bật) → xuất artifacts.

**Lưu ý:** chạy hết notebook này sẽ tốn RAM/thời gian, và ghi đè outputs cũ.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


Project: D:\network-anomaly-hybrid-vscode
Dataset: D:\network-anomaly-hybrid-vscode\data\raw


In [2]:
print('SAMPLE:', cfg.SAMPLE, '| TEST_FRAC:', cfg.TEST_FRAC,
      '| WITH_SUPERVISED:', cfg.WITH_SUPERVISED)
print('Mục tiêu FPR:', cfg.FPR_TARGET)
print('Để đổi cấu hình, đặt biến môi trường trước khi mở kernel, rồi Restart Kernel.')


SAMPLE: 300000 | TEST_FRAC: 0.25 | WITH_SUPERVISED: True
Mục tiêu FPR: 0.01
Để đổi cấu hình, đặt biến môi trường trước khi mở kernel, rồi Restart Kernel.


In [3]:
from run_pipeline import main
main()


[17:57:35] Backed up D:\network-anomaly-hybrid-vscode\outputs to D:\network-anomaly-hybrid-vscode\outputs_backups\20261113_175720_570153
[17:57:36] Backed up D:\network-anomaly-hybrid-vscode\data\curated\cse2018_curated to D:\network-anomaly-hybrid-vscode\data\curated\cse2018_curated_backups\20261113_175735_295961
Train: ['2018-02-14', '2018-02-15'] Validation: ['2018-02-16'] Test: ['2018-02-20', '2018-02-21', '2018-02-22', '2018-02-23', '2018-02-28', '2018-03-01', '2018-03-02']
[17:57:59] Ingest and prepare parquet
[17:59:57] Benign training 1,268,743 -> 1,268,743 records (invalid cells replaced with zero)
[18:00:21] Using fixed test snapshot: D:\network-anomaly-hybrid-vscode\data\curated\test_snapshots\8de60e51fbb7c1c886dafe6d
[18:00:23] Training models on 300,035 benign records
[18:06:58] Đã huấn luyện IF + AE trên mẫu 300,035 x 46; AE (46, 46, 23, 11, 23, 46, 46)
[18:06:58] Models trained in 395.6s
[18:07:42] Calibrated threshold: {'s_iso': 0.5834286909621103, 's_ae': 2.07714828968

In [4]:
import json
import pandas as pd
from IPython.display import display
p = cfg.OUT_DIR / 'metrics' / 'metrics_by_day.csv'
if p.exists():
    display(pd.read_csv(p).round(4))
print('Models:', list((cfg.OUT_DIR / 'models').glob('*.joblib')))
print('Figures:', list((cfg.OUT_DIR / 'figures').glob('*.png')))


,model,day,n,attack_rate,recall,fpr,precision,f1,roc_auc,pr_auc
0,Isolation Forest,2018-02-20,238387,0.6029,0.0002,0.0301,0.0077,0.0003,0.7836,0.7084
1,Isolation Forest,2018-02-21,140593,0.3572,0.0001,0.0055,0.0139,0.0003,0.0171,0.2075
2,Isolation Forest,2018-02-22,207495,0.0004,0.0706,0.0251,0.0012,0.0023,0.7607,0.0010
3,Isolation Forest,2018-02-23,207352,0.0006,0.3937,0.0253,0.0095,0.0185,0.8401,0.0074
4,Isolation Forest,2018-02-28,113693,0.1236,0.0539,0.0536,0.1242,0.0752,0.5072,0.1249
5,Isolation Forest,2018-03-01,62155,0.2503,0.0748,0.0533,0.3190,0.1211,0.5117,0.2644
6,Isolation Forest,2018-03-02,193494,0.1872,0.0071,0.0329,0.0471,0.0123,0.8833,0.4475
7,Isolation Forest,ALL,1163169,0.2235,0.0087,0.0296,0.0779,0.0157,0.6589,0.2696
8,Autoencoder,2018-02-20,238387,0.6029,0.0014,0.0016,0.5587,0.0027,0.9612,0.9372
9,Autoencoder,2018-02-21,140593,0.3572,0.0090,0.0000,1.0000,0.0179,0.0092,0.2137


Models: [WindowsPath('D:/network-anomaly-hybrid-vscode/outputs/models/feature_spec.joblib'), WindowsPath('D:/network-anomaly-hybrid-vscode/outputs/models/model.joblib')]
Figures: [WindowsPath('D:/network-anomaly-hybrid-vscode/outputs/figures/ae_loss.png'), WindowsPath('D:/network-anomaly-hybrid-vscode/outputs/figures/metrics_by_day.png')]
